In [1]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CATEGORIES
from src.utils.io import read_jsonl_sample

CATEGORY = 'fantasy_paranormal'
cfg = CATEGORIES[CATEGORY]
OUTPUT_DIR = cfg.processed_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Category: {cfg.display_name}')
print(f'Books file: {cfg.books_file}')
print(f'Interactions file: {cfg.interactions_file}')
print(f'Output dir: {OUTPUT_DIR}')


Category: Fantasy & Paranormal
Books file: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\raw\goodreads_books_fantasy_paranormal.json.gz
Interactions file: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\raw\goodreads_interactions_fantasy_paranormal.json.gz
Output dir: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\interim\fantasy_paranormal


---
# Books

In [2]:
SAMPLE_SIZE = 5_000
books_sample = read_jsonl_sample(cfg.books_file, nrows=SAMPLE_SIZE)
print(f'Shape: {books_sample.shape}')
print(f'Columns ({len(books_sample.columns)}):')
print(list(books_sample.columns))

Shape: (5000, 29)
Columns (29):
['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code', 'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin', 'similar_books', 'description', 'format', 'link', 'authors', 'publisher', 'num_pages', 'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'work_id', 'title', 'title_without_series']


In [3]:
summary_books = pd.DataFrame({
    'dtype': books_sample.dtypes.astype(str),
    'non_null': books_sample.notna().sum(),
    'null_count': books_sample.isna().sum(),
    'null_pct': (books_sample.isna().sum() / len(books_sample) * 100).round(1),
    'n_unique': books_sample.apply(lambda col: col.map(str).nunique()),
    'example': books_sample.iloc[0].astype(str).str[:80],
})
display(summary_books)

,dtype,non_null,null_count,null_pct,n_unique,example
isbn,str,5000,0,0.0,2527,
text_reviews_count,str,5000,0,0.0,305,7
series,object,5000,0,0.0,3328,['189911']
country_code,str,5000,0,0.0,1,US
language_code,str,5000,0,0.0,44,eng
popular_shelves,object,5000,0,0.0,4655,"[{'count': '58', 'name': 'to-read'}, {'count':..."
asin,str,5000,0,0.0,1322,B00071IKUY
is_ebook,str,5000,0,0.0,2,false
average_rating,str,5000,0,0.0,212,4.03
kindle_asin,str,5000,0,0.0,2346,


In [4]:
books_sample.head(3)

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,,7,[189911],US,eng,"[{'count': '58', 'name': 'to-read'}, {'count':...",B00071IKUY,false,4.03,,...,,Book Club Edition,1987,https://www.goodreads.com/book/show/7327624-th...,https://images.gr-assets.com/books/1304100136m...,7327624,140,8948723,"The Unschooled Wizard (Sun Wolf and Starhawk, ...","The Unschooled Wizard (Sun Wolf and Starhawk, ..."
1,1934876569,6,[151854],US,,"[{'count': '515', 'name': 'to-read'}, {'count'...",,false,4.22,,...,3,,2009,https://www.goodreads.com/book/show/6066812-al...,https://images.gr-assets.com/books/1316637798m...,6066812,98,701117,All's Fairy in Love and War (Avalon: Web of Ma...,All's Fairy in Love and War (Avalon: Web of Ma...
2,,60,[1052227],US,eng,"[{'count': '54', 'name': 'currently-reading'},...",B01NCIKAQX,true,4.33,B01NCIKAQX,...,,,,https://www.goodreads.com/book/show/33394837-t...,https://images.gr-assets.com/books/1493114742m...,33394837,269,54143148,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2)


In [5]:
BOOKS_COLUMNS_TO_DROP = [
    'isbn',
    'isbn13',
    'asin',
    'kindle_asin',
    'url',
    'link',
    'image_url',
    'country_code',
    'edition_information',
    'publication_day',
    'publication_month',
    'similar_books',
    'title_without_series',
    'is_ebook',
    'description',
    'publisher',
    'format'
]

In [6]:
books_clean = books_sample.drop(columns=[c for c in BOOKS_COLUMNS_TO_DROP if c in books_sample.columns])

In [7]:
summary_clean = pd.DataFrame({
    'dtype': books_clean.dtypes.astype(str),
    'non_null': books_clean.notna().sum(),
    'null_pct': (books_clean.isna().sum() / len(books_clean) * 100).round(1),
})
display(summary_clean)

,dtype,non_null,null_pct
text_reviews_count,str,5000,0.0
series,object,5000,0.0
language_code,str,5000,0.0
popular_shelves,object,5000,0.0
average_rating,str,5000,0.0
authors,object,5000,0.0
num_pages,str,5000,0.0
publication_year,str,5000,0.0
book_id,str,5000,0.0
ratings_count,str,5000,0.0


In [8]:
books_clean.head(5)

,text_reviews_count,series,language_code,popular_shelves,average_rating,authors,num_pages,publication_year,book_id,ratings_count,work_id,title
0,7,[189911],eng,"[{'count': '58', 'name': 'to-read'}, {'count':...",4.03,"[{'author_id': '10333', 'role': ''}]",600,1987,7327624,140,8948723,"The Unschooled Wizard (Sun Wolf and Starhawk, ..."
1,6,[151854],,"[{'count': '515', 'name': 'to-read'}, {'count'...",4.22,"[{'author_id': '19158', 'role': ''}]",216,2009,6066812,98,701117,All's Fairy in Love and War (Avalon: Web of Ma...
2,60,[1052227],eng,"[{'count': '54', 'name': 'currently-reading'},...",4.33,"[{'author_id': '242185', 'role': ''}]",318,,33394837,269,54143148,The House of Memory (Pluto's Snitch #2)
3,1,[147734],,"[{'count': '1057', 'name': 'to-read'}, {'count...",4.04,"[{'author_id': '50873', 'role': ''}, {'author_...",,,12182387,4,285263,"The Passion (Dark Visions, #3)"
4,21,[811663],en-US,"[{'count': '598', 'name': 'to-read'}, {'count'...",4.23,"[{'author_id': '5360266', 'role': ''}]",,,29074693,149,46079519,"Prowled Darkness (Dante's Circle, #7)"


In [9]:
from src.utils.io import read_jsonl_chunks
import pyarrow as pa
import pyarrow.parquet as pq

BOOKS_OUTPUT_PATH = OUTPUT_DIR / 'books_clean.parquet'
CHUNKSIZE = 50_000

writer = None
total_rows = 0

for chunk_idx, chunk in enumerate(read_jsonl_chunks(cfg.books_file, chunksize=CHUNKSIZE)):
    chunk = chunk.drop(columns=[c for c in BOOKS_COLUMNS_TO_DROP if c in chunk.columns])
    total_rows += len(chunk)
    
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(BOOKS_OUTPUT_PATH, table.schema, compression='snappy')
    writer.write_table(table)
    print(f'  Chunk {chunk_idx}: {len(chunk):,} rows (total: {total_rows:,})')

if writer:
    writer.close()

  Chunk 0: 50,000 rows (total: 50,000)
  Chunk 1: 50,000 rows (total: 100,000)
  Chunk 2: 50,000 rows (total: 150,000)
  Chunk 3: 50,000 rows (total: 200,000)
  Chunk 4: 50,000 rows (total: 250,000)
  Chunk 5: 8,585 rows (total: 258,585)


In [10]:
metadata = pq.read_metadata(BOOKS_OUTPUT_PATH)
print(f'Rows parquet: {metadata.num_rows:,}')
print(f'Columns: {metadata.num_columns}')
print(f'File size: {BOOKS_OUTPUT_PATH.stat().st_size / 1e6:.1f} MB')

df_check = pd.read_parquet(BOOKS_OUTPUT_PATH, columns=None).head(3)
print(f'\nFinal columns ({len(df_check.columns)}):')
print(list(df_check.columns))
display(df_check)

Rows parquet: 258,585
Columns: 14
File size: 172.0 MB

Final columns (12):
['text_reviews_count', 'series', 'language_code', 'popular_shelves', 'average_rating', 'authors', 'num_pages', 'publication_year', 'book_id', 'ratings_count', 'work_id', 'title']


,text_reviews_count,series,language_code,popular_shelves,average_rating,authors,num_pages,publication_year,book_id,ratings_count,work_id,title
0,7,[189911],eng,"[{'count': '58', 'name': 'to-read'}, {'count':...",4.03,"[{'author_id': '10333', 'role': ''}]",600,1987,7327624,140,8948723,"The Unschooled Wizard (Sun Wolf and Starhawk, ..."
1,6,[151854],,"[{'count': '515', 'name': 'to-read'}, {'count'...",4.22,"[{'author_id': '19158', 'role': ''}]",216,2009,6066812,98,701117,All's Fairy in Love and War (Avalon: Web of Ma...
2,60,[1052227],eng,"[{'count': '54', 'name': 'currently-reading'},...",4.33,"[{'author_id': '242185', 'role': ''}]",318,,33394837,269,54143148,The House of Memory (Pluto's Snitch #2)


# Interactions

In [11]:
INTER_SAMPLE_SIZE = 10_000
inter_sample = read_jsonl_sample(cfg.interactions_file, nrows=INTER_SAMPLE_SIZE)
print(f'Shape: {inter_sample.shape}')
print(f'Columns ({len(inter_sample.columns)}):')
print(list(inter_sample.columns))

Shape: (10000, 10)
Columns (10):
['user_id', 'book_id', 'review_id', 'is_read', 'rating', 'review_text_incomplete', 'date_added', 'date_updated', 'read_at', 'started_at']


In [12]:
summary_inter = pd.DataFrame({
    'dtype': inter_sample.dtypes.astype(str),
    'non_null': inter_sample.notna().sum(),
    'null_count': inter_sample.isna().sum(),
    'null_pct': (inter_sample.isna().sum() / len(inter_sample) * 100).round(1),
    'n_unique': inter_sample.apply(lambda col: col.map(str).nunique()),
    'example': inter_sample.iloc[0].astype(str).str[:80],
})
display(summary_inter)

,dtype,non_null,null_count,null_pct,n_unique,example
user_id,str,10000,0,0.0,91,8842281e1d1347389f2ab93d60773d4d
book_id,str,10000,0,0.0,6599,19161852
review_id,str,10000,0,0.0,10000,4443cb6883624c3772625ef5b7b4e138
is_read,bool,10000,0,0.0,2,False
rating,int64,10000,0,0.0,6,0
review_text_incomplete,str,10000,0,0.0,497,
date_added,str,10000,0,0.0,9519,Fri Sep 08 10:44:24 -0700 2017
date_updated,str,10000,0,0.0,9472,Fri Sep 08 10:44:24 -0700 2017
read_at,str,10000,0,0.0,1457,
started_at,str,10000,0,0.0,1222,


In [13]:
inter_sample.head(3)

,user_id,book_id,review_id,is_read,rating,review_text_incomplete,date_added,date_updated,read_at,started_at
0,8842281e1d1347389f2ab93d60773d4d,19161852,4443cb6883624c3772625ef5b7b4e138,False,0,,Fri Sep 08 10:44:24 -0700 2017,Fri Sep 08 10:44:24 -0700 2017,,
1,8842281e1d1347389f2ab93d60773d4d,18245960,dfdbb7b0eb5a7e4c26d59a937e2e5feb,True,5,This is a special book. It started slow for ab...,Sun Jul 30 07:44:10 -0700 2017,Wed Aug 30 00:00:26 -0700 2017,Sat Aug 26 12:05:52 -0700 2017,Tue Aug 15 13:23:18 -0700 2017
2,8842281e1d1347389f2ab93d60773d4d,32075825,11ffeb4204d7421f716a8f91c190ef2c,False,0,,Wed May 31 06:41:50 -0700 2017,Wed May 31 06:41:51 -0700 2017,,


In [14]:
INTER_COLUMNS_TO_DROP = [
    'review_text_incomplete',
    'review_id',
    'read_at',
    'started_at',
]

In [15]:
inter_clean = inter_sample.drop(columns=[c for c in INTER_COLUMNS_TO_DROP if c in inter_sample.columns])

In [16]:
summary_inter_clean = pd.DataFrame({
    'dtype': inter_clean.dtypes.astype(str),
    'non_null': inter_clean.notna().sum(),
    'null_pct': (inter_clean.isna().sum() / len(inter_clean) * 100).round(1),
})
display(summary_inter_clean)

,dtype,non_null,null_pct
user_id,str,10000,0.0
book_id,str,10000,0.0
is_read,bool,10000,0.0
rating,int64,10000,0.0
date_added,str,10000,0.0
date_updated,str,10000,0.0


In [17]:
inter_clean.head(5)

,user_id,book_id,is_read,rating,date_added,date_updated
0,8842281e1d1347389f2ab93d60773d4d,19161852,False,0,Fri Sep 08 10:44:24 -0700 2017,Fri Sep 08 10:44:24 -0700 2017
1,8842281e1d1347389f2ab93d60773d4d,18245960,True,5,Sun Jul 30 07:44:10 -0700 2017,Wed Aug 30 00:00:26 -0700 2017
2,8842281e1d1347389f2ab93d60773d4d,32075825,False,0,Wed May 31 06:41:50 -0700 2017,Wed May 31 06:41:51 -0700 2017
3,8842281e1d1347389f2ab93d60773d4d,43615,False,0,Mon Apr 03 13:27:29 -0700 2017,Mon Apr 03 13:27:30 -0700 2017
4,8842281e1d1347389f2ab93d60773d4d,26721984,False,0,Wed Mar 29 00:28:30 -0700 2017,Wed Mar 29 00:28:31 -0700 2017


In [18]:
from src.utils.io import read_jsonl_chunks
import pyarrow as pa
import pyarrow.parquet as pq

INTER_OUTPUT_PATH = OUTPUT_DIR / 'interactions_clean.parquet'
INTER_CHUNKSIZE = 500_000

writer = None
total_rows = 0

for chunk_idx, chunk in enumerate(read_jsonl_chunks(cfg.interactions_file, chunksize=INTER_CHUNKSIZE)):
    chunk = chunk.drop(columns=[c for c in INTER_COLUMNS_TO_DROP if c in chunk.columns])
    total_rows += len(chunk)
    
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(INTER_OUTPUT_PATH, table.schema, compression='snappy')
    writer.write_table(table)
    
    if chunk_idx % 10 == 0:
        print(f'  Chunk {chunk_idx}: {total_rows:,} rows so far')

if writer:
    writer.close()

print(f'   Total rows: {total_rows:,}')

  Chunk 0: 500,000 rows so far
  Chunk 10: 5,500,000 rows so far
  Chunk 20: 10,500,000 rows so far
  Chunk 30: 15,500,000 rows so far
  Chunk 40: 20,500,000 rows so far
  Chunk 50: 25,500,000 rows so far
  Chunk 60: 30,500,000 rows so far
  Chunk 70: 35,500,000 rows so far
  Chunk 80: 40,500,000 rows so far
  Chunk 90: 45,500,000 rows so far
  Chunk 100: 50,500,000 rows so far
  Chunk 110: 55,397,550 rows so far
   Total rows: 55,397,550


In [19]:
metadata = pq.read_metadata(INTER_OUTPUT_PATH)
print(f'Rows parquet: {metadata.num_rows:,}')
print(f'Columns: {metadata.num_columns}')
print(f'File size: {INTER_OUTPUT_PATH.stat().st_size / 1e6:.1f} MB')

df_check = pd.read_parquet(INTER_OUTPUT_PATH).head(3)
print(f'\nFinal columns ({len(df_check.columns)}):')
print(list(df_check.columns))
display(df_check)

Rows parquet: 55,397,550
Columns: 6
File size: 1312.2 MB

Final columns (6):
['user_id', 'book_id', 'is_read', 'rating', 'date_added', 'date_updated']


,user_id,book_id,is_read,rating,date_added,date_updated
0,8842281e1d1347389f2ab93d60773d4d,19161852,False,0,Fri Sep 08 10:44:24 -0700 2017,Fri Sep 08 10:44:24 -0700 2017
1,8842281e1d1347389f2ab93d60773d4d,18245960,True,5,Sun Jul 30 07:44:10 -0700 2017,Wed Aug 30 00:00:26 -0700 2017
2,8842281e1d1347389f2ab93d60773d4d,32075825,False,0,Wed May 31 06:41:50 -0700 2017,Wed May 31 06:41:51 -0700 2017
